# FinSight360 — Phase 3: Anomaly Detection EDA

Exploratory analysis of:
1. Benford's Law conformity across companies
2. Isolation Forest anomaly scores
3. Feature distributions and correlations
4. SHAP feature importance

In [ ]:
import os
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Connect to the shared DuckDB file
DB_PATH = os.environ.get('DUCKDB_PATH', '../data/finsight360.duckdb')
conn = duckdb.connect(DB_PATH, read_only=True)
print(f'Connected to {DB_PATH}')

## 1. Risk Distribution Overview

In [ ]:
# Load ML scores
try:
    scores = conn.execute('SELECT * FROM ml_anomaly_scores ORDER BY ml_risk_score DESC').df()
    print(f'ML scores: {len(scores)} companies')
    print(scores[['ticker', 'ml_risk_score', 'anomaly_percentile', 'risk_tier', 'is_anomaly']].head(10))
except Exception as e:
    print(f'ML scores not found (run ml-run first): {e}')
    scores = pd.DataFrame()

In [ ]:
if not scores.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Distribution of ML risk scores
    axes[0].hist(scores['ml_risk_score'], bins=20, color='steelblue', edgecolor='white')
    axes[0].axvline(70, color='red', linestyle='--', label='High-risk threshold (70)')
    axes[0].set_xlabel('ML Risk Score')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Distribution of ML Risk Scores')
    axes[0].legend()

    # Risk tier breakdown
    tier_counts = scores['risk_tier'].value_counts()
    colors = {'HIGH': '#d32f2f', 'MEDIUM': '#f57c00', 'LOW': '#fbc02d', 'CLEAN': '#388e3c'}
    bar_colors = [colors.get(t, 'gray') for t in tier_counts.index]
    axes[1].bar(tier_counts.index, tier_counts.values, color=bar_colors)
    axes[1].set_xlabel('Risk Tier')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Risk Tier Distribution')

    plt.tight_layout()
    plt.show()

## 2. Benford's Law Analysis

In [ ]:
from ml.benford import BenfordAnalyzer, BENFORD_DISTRIBUTION
import sys; sys.path.insert(0, '..')

try:
    benford = conn.execute('SELECT * FROM benford_results ORDER BY benford_risk_score DESC').df()
    print(benford[['ticker', 'n_observations', 'mad_score', 'conformity_rating', 'benford_risk_score']])
except Exception as e:
    print(f'Benford results not found: {e}')
    benford = pd.DataFrame()

In [ ]:
# Plot Benford's expected vs observed for a single company
try:
    facts = conn.execute("""
        SELECT ff.value FROM financial_facts ff
        JOIN companies c ON ff.cik = c.cik
        WHERE c.ticker = 'AAPL' AND ff.value != 0
    """).df()

    analyzer = BenfordAnalyzer()
    result = analyzer.analyze_series(facts['value'], 'AAPL', 'Apple Inc.', 'AAPL')

    digits = list(range(1, 10))
    observed = [result.observed_frequencies.get(d, 0) for d in digits]
    expected = [BENFORD_DISTRIBUTION[d] for d in digits]

    x = np.arange(len(digits))
    width = 0.35
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width/2, observed, width, label='Observed', color='steelblue')
    ax.bar(x + width/2, expected, width, label="Benford's Expected", color='orange')
    ax.set_xlabel('Leading Digit')
    ax.set_ylabel('Frequency')
    ax.set_title(f"Benford's Law — AAPL (MAD={result.mad_score:.4f}, {result.conformity_rating})")
    ax.set_xticks(x)
    ax.set_xticklabels(digits)
    ax.legend()
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Plot failed: {e}')

## 3. Feature Correlations

In [ ]:
from ml.feature_builder import ISOLATION_FOREST_FEATURES

if not scores.empty:
    avail = [f for f in ISOLATION_FOREST_FEATURES if f in scores.columns]
    if avail:
        corr = scores[avail].corr()
        fig, ax = plt.subplots(figsize=(12, 10))
        sns.heatmap(corr, annot=False, cmap='RdBu_r', center=0, ax=ax)
        ax.set_title('Feature Correlation Matrix')
        plt.tight_layout()
        plt.show()

## 4. Rule-based vs ML Risk Scores

In [ ]:
if not scores.empty and 'pre_ml_risk_score' in scores.columns:
    fig = px.scatter(
        scores,
        x='pre_ml_risk_score',
        y='ml_risk_score',
        color='risk_tier',
        hover_data=['ticker', 'company_name'],
        title='Rule-based vs ML Risk Scores',
        labels={
            'pre_ml_risk_score': 'Rule-based Risk Score (dbt)',
            'ml_risk_score': 'ML Risk Score (Isolation Forest)'
        },
        color_discrete_map={
            'HIGH': 'red', 'MEDIUM': 'orange', 'LOW': 'gold', 'CLEAN': 'green'
        }
    )
    fig.show()

In [ ]:
conn.close()
print('Analysis complete.')